# 03. Statistical analysis

**Owner:** Anibal  \
**Inputs:** see `config/paths.yml`  \
**Outputs:** figures to `reports/figures/`, data to `data/processed/`

## Purpose

Question A: Are demographic ageing and rural depopulation associated  with wildfire incidence and with burned-area impact at the municipality level?


## Setup

In [ ]:
import sys
import warnings
from pathlib import Path

# Make the src-layout package importable when the notebook kernel is not installed with -e .
project_root = Path.cwd()
while project_root != project_root.parent and not (project_root / "pyproject.toml").is_file():
    project_root = project_root.parent
sys.path.insert(0, str(project_root / "src"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm

from wildfires.config import STUDY_YEARS
from wildfires.io import load_icnf, load_panel
from wildfires.viz import apply_theme

apply_theme()
warnings.filterwarnings("ignore", category=FutureWarning)

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 140)

## Load panel

### INE (population dataset)

In [ ]:
panel = load_panel()
panel = panel.loc[
    panel["year"].between(STUDY_YEARS[0], STUDY_YEARS[1])
].copy()
panel["share_over_65"] = panel["share_65_plus"] / 100
panel["share_over_75"] = panel["share_75_plus"] / 100
panel.head()

In [ ]:
panel

### INCF (wildfire dataset)

In [ ]:
wildfire_data = load_icnf(level="concelho")
wildfire_data.head()

### Municipality dimensions (data.gov)

In [ ]:
municipality_dimensions = (
    panel[["dtcc", "territory", "municipality_area_km2"]]
    .drop_duplicates("dtcc")
    .rename(columns={"territory": "municipality", "municipality_area_km2": "area_km2"})
)
municipality_dimensions.head()

## Exploratory visualisations

### Fact checking the metrics - Study case: Penedono

In [ ]:
pop_penedono = panel.loc[
    panel["territory"].eq("Penedono"),
    ["pop_0_14", "pop_15_24", "pop_25_64", "pop_65_plus", "pop_75_plus", "year"],
].sort_values("year")

In [ ]:
pop_penedono

In [ ]:

# Set year as index.
df = pop_penedono.set_index("year")

plt.figure(figsize=(10, 6))

population_cols = ["pop_0_14", "pop_15_24", "pop_25_64", "pop_65_plus", "pop_75_plus"]
for col in population_cols:
    plt.plot(df.index, df[col], marker="o", label=col)

plt.title("Population by Age Group - Penedono")
plt.xlabel("Year")
plt.ylabel("Population")
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
wildfire_penedono = wildfire_data.loc[
    wildfire_data["year"].between(2001, 2025)
    & wildfire_data["Concelho"].eq("Penedono"),
    ["year", "Num_IncendiosRurais"],
].rename(columns={"Num_IncendiosRurais": "n_fires"})

In [ ]:
import matplotlib.pyplot as plt

pop_df = pop_penedono.set_index("year")
fire_df = wildfire_penedono.set_index("year")

fig, ax1 = plt.subplots(figsize=(12, 7))

population_cols = ["pop_0_14", "pop_15_24", "pop_25_64", "pop_65_plus", "pop_75_plus"]
for col in population_cols:
    ax1.plot(pop_df.index, pop_df[col], marker="o", linewidth=2, label=col)

ax1.set_xlabel("Year")
ax1.set_ylabel("Population")
ax1.grid(True, alpha=0.3)

ax2 = ax1.twinx()
ax2.plot(
    fire_df.index,
    fire_df["n_fires"],
    color="red",
    linewidth=5,
    alpha=0.8,
    label="Number of Rural Fires",
)
ax2.set_ylabel("Number of Rural Fires", color="red")
ax2.tick_params(axis="y", labelcolor="red")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

plt.title("Population Structure and Rural Fires - Penedono")
plt.tight_layout()
plt.show()

**Conclusion:** Data shows a specific spike in 2021 both in ageing of the population and number of rural fires which may indicate the a link between the two variables or simply chance. However the number of fires was low in 2025, while in reality, the municipality of Penedono was devastated in 2025. Therefore, we conclude that the number of fires may not represent a good metric to represent the damage. Let's investigate other metrics.

### Burned area ratio - Penedono

In this chapter, we investigate the burned area ratio as a severity measure. This metric has been used in scientific research related to socio-economic impact (Chas-Amil et al., 2022) [@chas_amil2022], which is seems adequate to our research question.

In [ ]:
# Build one municipality-year dataset for burned-area ratio and population ageing.
fire_area = wildfire_data[
    ["dtcc", "year", "AreaArdTotal_IncendioInicioConc", "AreaArdTotal_NoConcelho"]
].copy()

fire_area["burned_area_ha"] = pd.to_numeric(
    fire_area["AreaArdTotal_IncendioInicioConc"], errors="coerce"
).fillna(pd.to_numeric(fire_area["AreaArdTotal_NoConcelho"], errors="coerce"))
fire_area = (
    fire_area.groupby(["dtcc", "year"], as_index=False, dropna=False)["burned_area_ha"]
    .sum(min_count=1)
)

municipality_area = municipality_dimensions.copy()
municipality_area["municipality_area_ha"] = municipality_area["area_km2"] * 100

older_population = panel[
    ["dtcc", "year", "pop_65_plus", "pop_75_plus"]
].rename(
    columns={
        "pop_65_plus": "population_over_65",
        "pop_75_plus": "population_over_75",
    }
)

burned_area_ratio = (
    fire_area
    .merge(municipality_area, on="dtcc", how="left", validate="many_to_one")
    .merge(older_population, on=["dtcc", "year"], how="left", validate="one_to_one")
)
burned_area_ratio["burned_area_ratio"] = (
    burned_area_ratio["burned_area_ha"] / burned_area_ratio["municipality_area_ha"]
)
burned_area_ratio["burned_area_ratio_pct"] = burned_area_ratio["burned_area_ratio"] * 100

burned_area_ratio = burned_area_ratio[
    [
        "dtcc", "municipality", "year", "burned_area_ha", "area_km2",
        "municipality_area_ha", "burned_area_ratio", "burned_area_ratio_pct",
        "population_over_65", "population_over_75",
    ]
].sort_values(["municipality", "year"]).reset_index(drop=True)

burned_area_ratio

In [ ]:
penedono_ratio = burned_area_ratio.loc[
    burned_area_ratio["municipality"].eq("Penedono"),
    ["year", "burned_area_ratio_pct", "population_over_65", "population_over_75"],
].sort_values("year")

fig, ax1 = plt.subplots(figsize=(12, 7))

ax1.plot(
    penedono_ratio["year"],
    penedono_ratio["burned_area_ratio_pct"],
    color="firebrick",
    marker="o",
    linewidth=2.5,
    label="Burned area ratio",
)
ax1.set_xlabel("Year")
ax1.set_ylabel("Burned area ratio (%)", color="firebrick")
ax1.tick_params(axis="y", labelcolor="firebrick")
ax1.grid(True, alpha=0.3)

ax2 = ax1.twinx()
ax2.plot(
    penedono_ratio["year"],
    penedono_ratio["population_over_65"],
    color="steelblue",
    marker="o",
    linewidth=2,
    label="Population over 65",
)
ax2.plot(
    penedono_ratio["year"],
    penedono_ratio["population_over_75"],
    color="darkorange",
    marker="o",
    linewidth=2,
    label="Population over 75",
)
ax2.set_ylabel("Older population (residents)")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

ax1.set_title("Burned Area Ratio and Demographic Ageing - Penedono")
fig.tight_layout()
plt.show()

**Conclusion:** The percentage of burned area seems to represent a better metric for the damage caused by the fire (spike in 2025 correspond to the catastrophic fires over the last year). However, we cannot conclude if the demographic ageing after 2020 have contributed to these fires in 2025. Therefore, we should verify if there is a pattern over the country.

### Burned area ratio - Country level

In [ ]:
# Build the country-level burned-area ratio and ageing dataset.
portugal_area = municipality_dimensions["area_km2"].sum()
portugal_area_ha = portugal_area * 100

country_burned_area = fire_area.groupby("year", as_index=False)["burned_area_ha"].sum(min_count=1)
country_burned_area["portugal_area_km2"] = portugal_area
country_burned_area["portugal_area_ha"] = portugal_area_ha
country_burned_area["burned_area_ratio"] = (
    country_burned_area["burned_area_ha"] / country_burned_area["portugal_area_ha"]
)
country_burned_area["burned_area_ratio_pct"] = country_burned_area["burned_area_ratio"] * 100

country_older_population = (
    panel.groupby("year", as_index=False)["pop_65_plus"]
    .sum(min_count=1)
    .rename(columns={"pop_65_plus": "population_over_65"})
)

portugal_burned_area_ratio = (
    country_burned_area
    .merge(country_older_population, on="year", how="left", validate="one_to_one")
    [[
        "year", "burned_area_ha", "portugal_area_km2", "portugal_area_ha",
        "burned_area_ratio", "burned_area_ratio_pct", "population_over_65",
    ]]
    .sort_values("year")
    .reset_index(drop=True)
)

portugal_burned_area_ratio

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 7))

ax1.plot(
    portugal_burned_area_ratio["year"],
    portugal_burned_area_ratio["burned_area_ratio_pct"],
    color="firebrick",
    marker="o",
    linewidth=2.5,
    label="Burned area ratio",
)
ax1.set_xlabel("Year")
ax1.set_ylabel("Burned area ratio (%)", color="firebrick")
ax1.tick_params(axis="y", labelcolor="firebrick")
ax1.grid(True, alpha=0.3)

ax2 = ax1.twinx()
ax2.plot(
    portugal_burned_area_ratio["year"],
    portugal_burned_area_ratio["population_over_65"],
    color="steelblue",
    marker="o",
    linewidth=2,
    label="Population over 65",
)
ax2.set_ylabel("Population over 65 (residents)", color="steelblue")
ax2.tick_params(axis="y", labelcolor="steelblue")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

ax1.set_title("Portugal Burned Area Ratio and Population Ageing")
fig.tight_layout()
plt.show()

**Conclusion:** Over the country we can observe ciclic patterns of fire damage incidence. The population over 65 is drastically increasing in last years. However, we do not have data about population demographics before 2019, which makes it hard to draw conclusions about the relationship between the two variables. We therefore proceed with additional analysis.

## Analysis panel

Let's build an analysis panel by selecting fire related variables, agricultural burned area and aggregating to municipality and year.

In [ ]:
# Panel: demographic ageing, rural depopulation, and agricultural fire outcomes.
analysis_panel = panel[
    [
        "dtcc", "year", "territory", "n_fires", "burned_ha_agric",
        "municipality_area_km2", "pop_total", "pop_65_plus", "pop_75_plus",
    ]
].copy()
analysis_panel = analysis_panel.rename(
    columns={
        "territory": "municipality",
        "n_fires": "rural_fire_count",
        "burned_ha_agric": "agricultural_burned_area_ha",
        "municipality_area_km2": "area_km2",
        "pop_total": "population_total",
        "pop_65_plus": "population_over_65",
        "pop_75_plus": "population_over_75",
    }
)
analysis_panel

Now we define population growth as a proxy to measure rural depopulation: Population growth = Population_t - Population_t-1 / Population_t-1

This allows to capture demographic change rather than population size.

In [ ]:
analysis_panel["municipality_area_ha"] = analysis_panel["area_km2"] * 100
analysis_panel["agri_burned_ratio_pct"] = (
    analysis_panel["agricultural_burned_area_ha"]
    / analysis_panel["municipality_area_ha"]
    * 100
)
analysis_panel["share_over_65"] = (
    analysis_panel["population_over_65"] / analysis_panel["population_total"]
)
analysis_panel["share_over_75"] = (
    analysis_panel["population_over_75"] / analysis_panel["population_total"]
)
analysis_panel = analysis_panel.sort_values(["dtcc", "year"]).reset_index(drop=True)
analysis_panel["population_growth"] = analysis_panel.groupby("dtcc")[
    "population_total"
].pct_change()
analysis_panel["log_fire_count"] = np.log1p(
    analysis_panel["rural_fire_count"].clip(lower=0)
)

analysis_panel = analysis_panel.replace([np.inf, -np.inf], np.nan)
analysis_panel

In [ ]:
plt.figure(figsize=(8,5))
plt.hist(
    analysis_panel["agri_burned_ratio_pct"].dropna(),
    bins=50,
    edgecolor="black"
)
plt.xlabel("Agricultural burned area ratio (%)")
plt.ylabel("Frequency")
plt.title("Distribution of agricultural burned area ratio")
plt.show()

As expected, it is extremely skewed. We need to do log transformation before proceeding.

In [ ]:
# use log1p because agricultural burned-area ratios are strongly right-skewed.
model_data = analysis_panel.copy()
model_data["log_agri_burned_ratio"] = np.log1p(
    model_data["agri_burned_ratio_pct"].clip(lower=0)
)

Now we can define municipality and year as fixed effects because of topography, climate, etc. Without these controls, differences between municipalities could bias the ageing coefficient.

In [ ]:
def fit_fixed_effects(data, predictors, outcome="log_agri_burned_ratio"):
    required = ["dtcc", "year", outcome, *predictors]
    sample = data[required].dropna().copy()
    design = sample[predictors].copy()
    design = pd.concat(
        [
            design,
            pd.get_dummies(sample["dtcc"], prefix="municipality", drop_first=True, dtype=float),
            pd.get_dummies(sample["year"], prefix="year", drop_first=True, dtype=float),
        ],
        axis=1,
    )
    design = sm.add_constant(design.astype(float), has_constant="add")
    fit = sm.OLS(sample[outcome].astype(float), design).fit(
        cov_type="cluster",
        cov_kwds={"groups": sample["dtcc"]},
    )
    return fit, sample, design

ageing_model, ageing_sample, ageing_design = fit_fixed_effects(
    model_data,
    ["share_over_65"],
)
ageing_growth_model, ageing_growth_sample, ageing_growth_design = fit_fixed_effects(
    model_data,
    ["share_over_65", "population_growth"],
)

model_comparison = pd.DataFrame(
    {
        "model": ["Ageing + municipality/year effects", "Ageing + population growth + effects"],
        "observations": [len(ageing_sample), len(ageing_growth_sample)],
        "r_squared": [ageing_model.rsquared, ageing_growth_model.rsquared],
        "share_over_65_coef": [
            ageing_model.params["share_over_65"],
            ageing_growth_model.params["share_over_65"],
        ],
        "share_over_65_pvalue": [
            ageing_model.pvalues["share_over_65"],
            ageing_growth_model.pvalues["share_over_65"],
        ],
        "population_growth_coef": [
            np.nan,
            ageing_growth_model.params["population_growth"],
        ],
        "population_growth_pvalue": [
            np.nan,
            ageing_growth_model.pvalues["population_growth"],
        ],
    }
)
model_comparison

The coefficient on share_over_65 is 0.669 and statistically significant at the 5% level (p = 0.038). This means a municipality whose elderly increases from 20% to 21% is associated with roughly a 0.7% increase in agricultural burned-area intensity. However, adding population growth, the ageing coef remains positive but significance weakens. Adding population growth only increases r squared modestly (0.205 to 0.230).

**Conclusion:** Overall, the results provide limited evidence that demographic structure is associated with agricultural fire outcomes, although the magnitude and statistical significance of the effects are modest.

The demographic panel is limited to 2019-2024. Let's forecasts Portugal's annual burned-area ratio using only its own history.

## Fire-only time-series forecasting

The 2001-2022 period is used for model development and 2023-2025 is held out for evaluation.

### Define and validate the target

In [ ]:
country_series = (
    portugal_burned_area_ratio.set_index("year")["burned_area_ratio_pct"]
    .sort_index()
    .astype(float)
)
expected_years = pd.Index(range(2001, 2026), name="year")
assert country_series.index.equals(expected_years)
assert country_series.notna().all()

country_series.to_frame("burned_area_ratio_pct").head()

### Chronological train-test split

The model is trained on 2001-2022 and evaluated on the untouched 2023-2025 period. This prevents later observations from influencing model selection and tests whether past fire ratios could anticipate recent years.

In [ ]:
train = country_series.loc[:2022]
test = country_series.loc[2023:2025]

split_summary = pd.DataFrame(
    {
        "period": ["training", "test"],
        "first_year": [train.index.min(), test.index.min()],
        "last_year": [train.index.max(), test.index.max()],
        "observations": [len(train), len(test)],
    }
)
split_summary

### Inspect autocorrelation and choose differencing

The raw series is non-negative and contains large spikes. We inspect the raw series and its first difference. The ACF/PACF are used as low-order ARIMA guides, not as an automatic high-order parameter search because annual data provides only 22 training observations.

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import adfuller

raw_adf = adfuller(train, autolag="AIC")
differenced_train = train.diff().dropna()
diff_adf = adfuller(differenced_train, autolag="AIC")

stationarity_checks = pd.DataFrame(
    {
        "series": ["level", "first_difference"],
        "ADF_statistic": [raw_adf[0], diff_adf[0]],
        "ADF_pvalue": [raw_adf[1], diff_adf[1]],
    }
)

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
axes[0, 0].plot(train.index, train, marker="o", color="firebrick")
axes[0, 0].set_title("Training series")
axes[0, 0].set_ylabel("Burned area ratio (%)")
axes[0, 1].plot(differenced_train.index, differenced_train, marker="o", color="darkgreen")
axes[0, 1].axhline(0, color="black", linewidth=0.8)
axes[0, 1].set_title("First difference")
axes[0, 1].set_ylabel("Change in ratio (percentage points)")
plot_acf(train, lags=8, ax=axes[1, 0], zero=False)
axes[1, 0].set_title("ACF: level series")
plot_acf(differenced_train, lags=8, ax=axes[1, 1], zero=False)
axes[1, 1].set_title("ACF: first difference")
fig.tight_layout()
plt.show()

stationarity_checks

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
plot_pacf(train, lags=8, ax=axes[0], method="ywm", zero=False)
axes[0].set_title("PACF: level series")
plot_pacf(differenced_train, lags=8, ax=axes[1], method="ywm", zero=False)
axes[1].set_title("PACF: first difference")
fig.tight_layout()
plt.show()

The level ACF and PACF show no strong low-order spike, so begin with a mean-only model and low-order ARMA alternatives. The differenced plots are retained as a sensitivity check, but they do not justify forcing d=1. Because there are only 22 training observations, model selection will use rolling one-step errors rather than a large ARIMA grid.

In [ ]:
arima_orders = {
    "ARIMA(0,0,0)": (0, 0, 0),
    "ARIMA(1,0,0)": (1, 0, 0),
    "ARIMA(0,0,1)": (0, 0, 1),
    "ARIMA(1,0,1)": (1, 0, 1),
}


def arima_forecast(series, order, steps=1, alpha=0.05):
    fitted = sm.tsa.ARIMA(series, order=order, trend="c").fit()
    result = fitted.get_forecast(steps=steps)
    mean = np.asarray(result.predicted_mean, dtype=float)
    interval = result.conf_int(alpha=alpha)
    lower = interval.iloc[:, 0].to_numpy(dtype=float)
    upper = interval.iloc[:, 1].to_numpy(dtype=float)
    return mean, lower, upper


def one_step_forecast(series, model_name):
    if model_name == "historical_mean":
        return float(series.mean())
    if model_name == "naive":
        return float(series.iloc[-1])
    return float(arima_forecast(series, arima_orders[model_name])[0][0])

candidate_models = ["historical_mean", "naive", *arima_orders]
candidate_models

### Select the model with rolling one-step validation

Each candidate is repeatedly fitted using data available up to year t and predicts year t+1. This imitates real forecasting and prevents choosing a model because it performs well only on the final holdout.

Let's use the rolling-origin time-series validation (or walk-forward validation). Basically, instead of fitting the model once, it reapeatedly use first observations to training history and predicts next observation. The expands the training series by one observation until the end.

In [ ]:
rolling_rows = []
for model_name in candidate_models:
    actual = []
    predicted = []
    for end_position in range(10, len(train)):
        history = train.iloc[:end_position]
        try:
            prediction = one_step_forecast(history, model_name)
        except (ValueError, np.linalg.LinAlgError):
            prediction = np.nan
        actual.append(float(train.iloc[end_position]))
        predicted.append(prediction)

    rolling_frame = pd.DataFrame({"actual": actual, "predicted": predicted}).dropna()
    rolling_rows.append(
        {
            "model": model_name,
            "validation_observations": len(rolling_frame),
            "MAE": (rolling_frame["predicted"] - rolling_frame["actual"]).abs().mean(),
            "RMSE": np.sqrt(
                ((rolling_frame["predicted"] - rolling_frame["actual"]) ** 2).mean()
            ),
        }
    )

rolling_scores = (
    pd.DataFrame(rolling_rows)
    .sort_values("RMSE")
    .reset_index(drop=True)
)
selected_model = rolling_scores.iloc[0]["model"]
rolling_scores

**Conclusion:** The best predictor is the long-run mean of the series. The simplest model performed best because data do not exhibit a strong predictable temporal pattern. Knowing previous years does not substantially improve prediction relative to simply using the long-run average.

Let's what happens visually.

### Forecast the 2023-2025 years

The selected model is refitted using all training observations through 2022. The final comparison keeps 2023-2025 untouched until this point. Prediction intervals show the range of values compatible with the model, not a guarantee that extreme fires will be covered.

In [ ]:
# forecast 2024-2025 using fire history
forecast_train = country_series.loc[:2023]
forecast_years = pd.Index([2024, 2025], name="year")

if selected_model == "historical_mean":
    forecast_values = np.repeat(forecast_train.mean(), len(forecast_years))
elif selected_model == "naive":
    forecast_values = np.repeat(forecast_train.iloc[-1], len(forecast_years))
else:
    forecast_values = arima_forecast(
        forecast_train,
        arima_orders[selected_model],
        steps=len(forecast_years),
    )[0]

forecast_2024_2025 = pd.DataFrame(
    {
        "actual": country_series.loc[forecast_years].to_numpy(),
        "predicted": forecast_values,
    },
    index=forecast_years,
)
forecast_2024_2025["absolute_error"] = (
    forecast_2024_2025["actual"] - forecast_2024_2025["predicted"]
).abs()

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(
    country_series.loc[:2023].index,
    country_series.loc[:2023],
    color="black",
    marker="o",
    label="Actual burned-area ratio",
)
ax.plot(
    forecast_2024_2025.index,
    forecast_2024_2025["actual"],
    color="firebrick",
    marker="o",
    linewidth=2.5,
    label="Actual 2024-2025",
)
ax.plot(
    [2023, *forecast_2024_2025.index],
    [country_series.loc[2023], *forecast_2024_2025["predicted"]],
    color="steelblue",
    marker="x",
    linestyle="--",
    linewidth=2,
    label=f"Predicted 2024-2025: {selected_model}",
)
ax.axvline(2023.5, color="grey", linestyle=":", label="Forecast origin")
ax.set_xlabel("Year")
ax.set_ylabel("Burned area ratio (%)")
ax.set_title("Actual versus Predicted Portugal Burned-Area Ratio")
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()
plt.show()

forecast_2024_2025

**Conclusion:** The 2025 increase was not predictable from the historical pattern contained in the series. The simple model assumed 2025 would be an "average" year. Instead, 2025 was much higher than average.

## Overall conclusion

The analysis explored whether population ageing and population decline are linked to agricultural fire outcomes.

Results suggest a possible relationship with burned area, but the evidence is not strong enough to draw firm conclusions. Municipalities with a higher share of older residents tended to have slightly higher agricultural burned areas, but the effect became weaker when population change was included in the model.

Importantly, the results were different for fire occurrence and fire impact:
* We found no clear link between demographic change and the number of fires.
* We found some indication that demographic change may be related to how much land burns when fires occur.

At the national level, past fire patterns were not very useful for forecasting future burned areas. A simple average-based forecast performed as well as more complex models, and it did not anticipate the large increase observed in 2025.

Overall, the current data does not provide strong evidence that ageing or depopulation directly increase fire risk, but it also does not rule out a relationship.

### Why the link may not be clear

Several factors may explain the inconclusive results:
* The demographic dataset covers only a short period (2019-2024).
* Effects of ageing and depopulation may take many years to appear.
* Key factors such as land abandonment, vegetation growth, weather conditions, and firefighting capacity were not directly measured.
* Fire outcomes are heavily influenced by drought, wind, and other environmental conditions that can outweigh demographic effects.
* Neighbor municipalities were not considered (e.g. a fire may pass from one municipality to the other but this relationship was not modeled).

### Key take aways
* Fire counts misrepresent damage; burned-area ratio is the defensible outcome measure.
* With municipality and year fixed effects, ageing carries a positive but marginal coefficient (0.669, p = 0.038) that weakens once population change is included.
* Demographic change shows no clear link to fire occurrence, and only limited evidence of a link to fire impact.
* National burned area is not predictable from its own history — the long-run mean beats every fitted time-series model, and 2025 was far above it.

### Recommendations for future analysis

A stronger assessment would require:
* Longer demographic and population histories.
* Better measures of rural decline and land abandonment.
* Weather and drought information.
* Land management and vegetation indicators.
* Testing delayed effects, where demographic change influences fire outcomes several years later.

With these additional data sources, it would be possible to determine more confidently whether demographic change contributes to agricultural fire risk or severity.